# Seizure Recognition with MLflow Model Logging
This notebook demonstrates training an EEG model with full MLflow integration including model logging, registration, and inference.

## 1. Environment & Imports

In [12]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import logging

import torch
import torch.nn as nn
import torch.optim as optim

import mlflow
import mlflow.pytorch

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"MLflow version: {mlflow.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.7.0+cu118
MLflow version: 3.13.0
CUDA available: False


## 2. MLflow Configuration

In [13]:
# Force MLflow tracking to localhost:5000
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
mlflow.set_tracking_uri("http://localhost:5000")

# Set experiment name
EXPERIMENT_NAME = "Seizure Recognition"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

MLflow Tracking URI: http://localhost:5000
Experiment: Seizure Recognition


## 3. Data Loading & Preprocessing

In [14]:
def load_epileptic_seizure_data(csv_path, seq_len=256, test_size=0.2):
    """Load real seizure data from CSV or generate synthetic."""
    if os.path.exists(csv_path):
        print(f"Loading CSV from {csv_path}")
        df = pd.read_csv(csv_path)
        # Assuming last column is label, rest are features
        X = df.iloc[:, :-1].values.astype(np.float32)
        y = df.iloc[:, -1].values.astype(np.int64)
        # Normalize each sample to mean 0, std 1
        X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-8)
        # Reshape to (batch, channels=1, seq_len) if needed
        if X.shape[1] != seq_len:
            print(f"Warning: CSV has {X.shape[1]} features, expected {seq_len}")
        X = X.reshape(X.shape[0], 1, -1)
    else:
        print(f"CSV not found at {csv_path}. Generating synthetic data.")
        X, y = generate_synthetic_eeg(n_samples=1000, seq_len=seq_len)
    
    # Split into train/val
    split = int(len(X) * (1 - test_size))
    X_train, X_val = X[:split], X[split:]
    y_train, y_val = y[:split], y[split:]
    
    return X_train, y_train, X_val, y_val


def generate_synthetic_eeg(n_samples=1000, seq_len=256, seizure_prob=0.2, seed=42):
    """Generate synthetic EEG signals with seizure patterns."""
    rng = np.random.RandomState(seed)
    X = rng.normal(0, 1, size=(n_samples, 1, seq_len)).astype(np.float32)
    y = np.zeros(n_samples, dtype=np.int64)
    for i in range(n_samples):
        if rng.rand() < seizure_prob:
            max_width = max(1, min(20, seq_len - 20))
            width = rng.randint(4, max_width + 1)
            pos_low = 10
            pos_high = seq_len - 10 - width + 1
            pos = pos_low if pos_high <= pos_low else rng.randint(pos_low, pos_high)
            amp = rng.uniform(3, 8)
            window = np.hanning(width)
            X[i, 0, pos:pos+width] += amp * window
            y[i] = 1
    return X, y

# Load data
csv_path = "/home/aiuser/app/Epileptic Seizure Recognition.csv"
seq_len = 256
X_train, y_train, X_val, y_val = load_epileptic_seizure_data(csv_path, seq_len=seq_len)

print(f"Training samples: {len(X_train)}, shape: {X_train.shape}")
print(f"Validation samples: {len(X_val)}, shape: {X_val.shape}")
print(f"Training labels distribution: {np.bincount(y_train)}")
print(f"Validation labels distribution: {np.bincount(y_val)}")

CSV not found at /home/aiuser/app/Epileptic Seizure Recognition.csv. Generating synthetic data.
Training samples: 800, shape: (800, 1, 256)
Validation samples: 200, shape: (200, 1, 256)
Training labels distribution: [622 178]
Validation labels distribution: [155  45]


## 4. Define Model Architecture

In [15]:
class SimpleEEGNet(nn.Module):
    def __init__(self, seq_len=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, padding=4),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(16, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

# Model hyperparameters
epochs = 10
lr = 1e-3
batch_size = 30

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleEEGNet(seq_len=seq_len).to(device)

print(f"Model initialized on device: {device}")
print(f"Model structure:\n{model}")

Model initialized on device: cpu
Model structure:
SimpleEEGNet(
  (net): Sequential(
    (0): Conv1d(1, 16, kernel_size=(9,), stride=(1,), padding=(4,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(16, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (4): ReLU()
    (5): AdaptiveAvgPool1d(output_size=1)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=32, out_features=1, bias=True)
  )
)


## 5. Training Loop & Evaluation

In [16]:
mlflow.pytorch.autolog()

# Start MLflow run
run = mlflow.start_run()
run_id = run.info.run_id
print(f"MLflow Run ID: {run_id}")

# Log hyperparameters
mlflow.log_params({
    "epochs": epochs,
    "lr": lr,
    "batch_size": batch_size,
    "seq_len": seq_len,
    "model_type": "SimpleEEGNet",
    "device": str(device),
})

loss_fn = nn.BCEWithLogitsLoss()
opt = optim.Adam(model.parameters(), lr=lr)

best_val_loss = float('inf')
training_history = {"train_loss": [], "val_loss": []}

# Training loop
for epoch in range(1, epochs + 1):
    # Train phase
    model.train()
    perm = np.random.permutation(len(X_train))
    train_losses = []
    
    for i in range(0, len(perm), batch_size):
        idx = perm[i:i+batch_size]
        xb = torch.from_numpy(X_train[idx]).to(device)
        yb = torch.from_numpy(y_train[idx].astype(np.float32)).to(device)
        
        opt.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()
        train_losses.append(loss.item())
    
    avg_train_loss = np.mean(train_losses)
    training_history["train_loss"].append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_losses = []
    with torch.no_grad():
        for i in range(0, len(X_val), batch_size):
            xb = torch.from_numpy(X_val[i:i+batch_size]).to(device)
            yb = torch.from_numpy(y_val[i:i+batch_size].astype(np.float32)).to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb).item()
            val_losses.append(loss)
    
    avg_val_loss = np.mean(val_losses)
    training_history["val_loss"].append(avg_val_loss)
    
    # Log metrics
    print(f"Epoch {epoch}/{epochs} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f}")
    mlflow.log_metric("train_loss", float(avg_train_loss), step=epoch)
    mlflow.log_metric("val_loss", float(avg_val_loss), step=epoch)
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"  ✓ New best validation loss: {best_val_loss:.4f}")

print(f"\nTraining complete. Best validation loss: {best_val_loss:.4f}")

MLflow Run ID: ed01db12dd61487dab250749e18818b8
Epoch 1/10 | train_loss=0.5684 | val_loss=0.5396
  ✓ New best validation loss: 0.5396
Epoch 2/10 | train_loss=0.5267 | val_loss=0.5211
  ✓ New best validation loss: 0.5211
Epoch 3/10 | train_loss=0.4971 | val_loss=0.4768
  ✓ New best validation loss: 0.4768
Epoch 4/10 | train_loss=0.4601 | val_loss=0.4242
  ✓ New best validation loss: 0.4242
Epoch 5/10 | train_loss=0.4126 | val_loss=0.3564
  ✓ New best validation loss: 0.3564
Epoch 6/10 | train_loss=0.3559 | val_loss=0.2915
  ✓ New best validation loss: 0.2915
Epoch 7/10 | train_loss=0.3136 | val_loss=0.2497
  ✓ New best validation loss: 0.2497
Epoch 8/10 | train_loss=0.2798 | val_loss=0.2184
  ✓ New best validation loss: 0.2184
Epoch 9/10 | train_loss=0.2532 | val_loss=0.1917
  ✓ New best validation loss: 0.1917
Epoch 10/10 | train_loss=0.2469 | val_loss=0.1872
  ✓ New best validation loss: 0.1872

Training complete. Best validation loss: 0.1872


## 6. Log Model to MLflow Model Registry

In [17]:
# Define output directory
output_dir = "models"
os.makedirs(output_dir, exist_ok=True)

# Save model locally
model_local_path = os.path.join(output_dir, "eeg_model_final.pt")
torch.save(model.state_dict(), model_local_path)
print(f"Model saved locally to {model_local_path}")

# Log model to MLflow using pytorch.log_model
# This creates an artifact in MLflow with the model
mlflow.pytorch.log_model(
    pytorch_model=model,
    artifact_path="pytorch_model",
    registered_model_name="eeg-seizure-detector"
)
print(f"✓ Model logged to MLflow with artifact path: pytorch_model")
print(f"✓ Model registered as: eeg-seizure-detector")

# Save metadata
metadata = {
    "mlflow_enabled": True,
    "run_id": run_id,
    "model_path_final": model_local_path,
    "best_val_loss": float(best_val_loss),
    "artifact_path": "pytorch_model",
    "registered_model_name": "eeg-seizure-detector"
}

meta_path = os.path.join(output_dir, "metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to {meta_path}")

2026/07/02 13:41:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Model saved locally to models/eeg_model_final.pt


2026/07/02 13:41:09 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/07/02 13:41:09 WARNING mlflow.utils.requirements_utils: Found torch version (2.7.0+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.7.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/02 13:41:11 WARNING mlflow.utils.requirements_utils: Found torch version (2.7.0+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.7.0' without the local version label 

✓ Model logged to MLflow with artifact path: pytorch_model
✓ Model registered as: eeg-seizure-detector
Metadata saved to models/metadata.json


Created version '2' of model 'eeg-seizure-detector'.


## 7. End MLflow Run

In [18]:
mlflow.end_run()
print("✓ MLflow run ended")
print(f"\nRun details:")
print(f"  Run ID: {run_id}")
print(f"  Experiment: {EXPERIMENT_NAME}")
print(f"  Best Val Loss: {best_val_loss:.4f}")

🏃 View run skittish-skink-239 at: http://localhost:5000/#/experiments/1/runs/ed01db12dd61487dab250749e18818b8
🧪 View experiment at: http://localhost:5000/#/experiments/1
✓ MLflow run ended

Run details:
  Run ID: ed01db12dd61487dab250749e18818b8
  Experiment: Seizure Recognition
  Best Val Loss: 0.1872


## 8. Load Model from MLflow and Run Inference

In [19]:
# Load model from MLflow using the run URI
model_uri = f"runs:/{run_id}/pytorch_model"
loaded_model = mlflow.pytorch.load_model(model_uri, map_location=device)
loaded_model.eval()
print(f"✓ Model loaded from MLflow URI: {model_uri}")

# Run inference on validation data
print("\nRunning inference on validation set...")
predictions = []
with torch.no_grad():
    for i in range(0, len(X_val), batch_size):
        xb = torch.from_numpy(X_val[i:i+batch_size]).to(device)
        logits = loaded_model(xb)
        preds = torch.sigmoid(logits).cpu().numpy()
        predictions.extend(preds)

predictions = np.array(predictions)
pred_labels = (predictions > 0.5).astype(int).flatten()

# Calculate metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_val, pred_labels)
precision = precision_score(y_val, pred_labels, zero_division=0)
recall = recall_score(y_val, pred_labels, zero_division=0)
f1 = f1_score(y_val, pred_labels, zero_division=0)
roc_auc = roc_auc_score(y_val, predictions)

print(f"Validation Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  ROC AUC:   {roc_auc:.4f}")

✓ Model loaded from MLflow URI: runs:/ed01db12dd61487dab250749e18818b8/pytorch_model

Running inference on validation set...
Validation Metrics:
  Accuracy:  0.9300
  Precision: 1.0000
  Recall:    0.6889
  F1 Score:  0.8158
  ROC AUC:   0.9855


## 9. Verify Model Predictions

In [20]:
# Compare original vs loaded model predictions on smaller sample
sample_size = 5
sample_x = torch.from_numpy(X_val[:sample_size]).to(device)

with torch.no_grad():
    original_pred = torch.sigmoid(model(sample_x)).cpu().numpy().flatten()
    loaded_pred = torch.sigmoid(loaded_model(sample_x)).cpu().numpy().flatten()

print("Sample predictions comparison (original vs loaded):")
print(f"True labels:        {y_val[:sample_size]}")
print(f"Original model:     {np.round(original_pred, 4)}")
print(f"Loaded model:       {np.round(loaded_pred, 4)}")
print(f"Predictions match:  {np.allclose(original_pred, loaded_pred)}")

Sample predictions comparison (original vs loaded):
True labels:        [0 1 0 1 0]
Original model:     [0.0239 0.503  0.0398 0.9657 0.0352]
Loaded model:       [0.0239 0.503  0.0398 0.9657 0.0352]
Predictions match:  True


## 10. Validation Tests

In [21]:
# Unit tests for model validation
print("Running validation tests...\n")

# Test 1: Model output shape
test_input = torch.randn(8, 1, seq_len).to(device)
output = loaded_model(test_input)
assert output.shape == (8,), f"Expected shape (8,), got {output.shape}"
print("✓ Test 1 passed: Model output shape is correct")

# Test 2: Predictions are in valid range
predictions_sample = torch.sigmoid(loaded_model(test_input)).detach().cpu().numpy()
assert np.all(predictions_sample >= 0) and np.all(predictions_sample <= 1), "Predictions out of range [0, 1]"
print("✓ Test 2 passed: Predictions are in valid range [0, 1]")

# Test 3: Model consistency
with torch.no_grad():
    pred1 = loaded_model(test_input)
    pred2 = loaded_model(test_input)
assert torch.allclose(pred1, pred2), "Model predictions not consistent"
print("✓ Test 3 passed: Model predictions are deterministic")

# Test 4: Validation metrics are reasonable
assert 0 <= accuracy <= 1, f"Accuracy out of range: {accuracy}"
assert 0 <= roc_auc <= 1, f"ROC AUC out of range: {roc_auc}"
print("✓ Test 4 passed: Validation metrics are in valid ranges")

print("\n✓ All validation tests passed!")

Running validation tests...

✓ Test 1 passed: Model output shape is correct
✓ Test 2 passed: Predictions are in valid range [0, 1]
✓ Test 3 passed: Model predictions are deterministic
✓ Test 4 passed: Validation metrics are in valid ranges

✓ All validation tests passed!


## 11. MLflow UI Instructions

To view your MLflow runs and models in the UI:

1. **Start MLflow UI** (run in terminal):
   ```bash
   mlflow ui --backend-store-uri sqlite:///mlruns.db --port 5000
   ```

2. **Access the UI**:
   - Open browser to: http://localhost:5000
   - View the "Seizure Recognition" experiment
   - See logged metrics, parameters, and artifacts

3. **View registered models**:
   - Navigate to "Models" section
   - Find "eeg-seizure-detector"
   - See model versions and stages

4. **Model file locations**:
   - Local model: `models/eeg_model_final.pt`
   - MLflow artifacts: `mlruns/<experiment_id>/<run_id>/artifacts/`
   - Metadata: `models/metadata.json`

## 12. Summary

In [22]:
print("="*60)
print("TRAINING & MLFLOW INTEGRATION SUMMARY")
print("="*60)
print(f"\n📊 Training Results:")
print(f"  - Epochs trained: {epochs}")
print(f"  - Best validation loss: {best_val_loss:.4f}")
print(f"  - Final validation accuracy: {accuracy:.4f}")
print(f"  - ROC AUC Score: {roc_auc:.4f}")

print(f"\n📁 Model Artifacts:")
print(f"  - Local path: {model_local_path}")
print(f"  - MLflow registered name: eeg-seizure-detector")
print(f"  - MLflow artifact path: pytorch_model")
print(f"  - Run ID: {run_id}")

print(f"\n🔬 Data:")
print(f"  - Training samples: {len(X_train)}")
print(f"  - Validation samples: {len(X_val)}")
print(f"  - Sequence length: {seq_len}")

print(f"\n✅ Notebook complete!")
print("="*60)

TRAINING & MLFLOW INTEGRATION SUMMARY

📊 Training Results:
  - Epochs trained: 10
  - Best validation loss: 0.1872
  - Final validation accuracy: 0.9300
  - ROC AUC Score: 0.9855

📁 Model Artifacts:
  - Local path: models/eeg_model_final.pt
  - MLflow registered name: eeg-seizure-detector
  - MLflow artifact path: pytorch_model
  - Run ID: ed01db12dd61487dab250749e18818b8

🔬 Data:
  - Training samples: 800
  - Validation samples: 200
  - Sequence length: 256

✅ Notebook complete!
